In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Environnement prêt !")

Environnement prêt !


## Activité 3.1 Inventaire des fichiers csv et identitification des variables 

In [6]:
# Dossier contenant les fichiers originaux
dossier_raw = Path("data/raw")

# Détection automatique des CSV
fichiers_csv = sorted(dossier_raw.glob("*.csv"))

# Construction de l'inventaire
inventaire = []

for fichier in fichiers_csv:
    df = pd.read_csv(fichier)

    inventaire.append({
        "Fichier": fichier.name,
        "Lignes": df.shape[0],
        "Colonnes": df.shape[1]
    })

df_inventaire = pd.DataFrame(inventaire)

display(df_inventaire)

,Fichier,Lignes,Colonnes
0,Sales 2020 T1.csv,24101,14
1,Sales 2020 T2.csv,25559,14
2,Sales 2020 T3.csv,27502,14
3,Sales 2021 T1.csv,20617,14
4,Sales 2021 T2.csv,23125,14
5,Sales 2021 T3.csv,23970,14


Commentaires : Notre jeu de données contient 6 fichiers CSV dont chacun contient 14 variables

### Activité 3.2
### Vérification des dates, des identifiants et les doublons

In [36]:
# Lecture d'un fichier pour examiner sa structure
df_exploration = pd.read_csv(fichiers_csv[1])

# Dimensions du fichier
print("Dimensions :", df_exploration.shape)

# Noms et types des variables
df_exploration.info()

# Aperçu des cinq premières observations
display(df_exploration.head())

Dimensions : (25559, 14)
<class 'pandas.DataFrame'>
RangeIndex: 25559 entries, 0 to 25558
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   IdDate                25559 non-null  str  
 1   IdVente               25559 non-null  int64
 2   IdBoutique            25559 non-null  int64
 3   IdProduit             25559 non-null  int64
 4   IdPromotion           25559 non-null  int64
 5   Montant du retour     25559 non-null  str  
 6   Quantité de retour    25559 non-null  int64
 7   Montant des ventes    25559 non-null  str  
 8   Quantité de vente     25559 non-null  int64
 9   Coût total            25559 non-null  str  
 10  Coût unitaire         25559 non-null  str  
 11  Prix unitaire         25559 non-null  str  
 12  Montant de la remise  25559 non-null  str  
 13  Quantité de remise    25559 non-null  int64
dtypes: int64(7), str(7)
memory usage: 2.7 MB


,IdDate,IdVente,IdBoutique,IdProduit,IdPromotion,Montant du retour,Quantité de retour,Montant des ventes,Quantité de vente,Coût total,Coût unitaire,Prix unitaire,Montant de la remise,Quantité de remise
0,4/1/2020 0:00,15004,230,1535,17,$268,1,"$3,216",12,"$1,355.64",$123.24,$268,$0,0
1,4/1/2020 0:00,18849,250,368,17,$0,0,"$15,588",12,"$5,164.56",$430.38,"$1,299",$0,0
2,4/1/2020 0:00,43763,208,243,17,$0,0,"$4,908",12,"$2,502.24",$208.52,$409,$0,0
3,4/1/2020 0:00,53457,230,809,17,$0,0,$478.80,12,$244.08,$20.34,$39.90,$0,0
4,4/1/2020 0:00,59735,207,1487,17,$0,0,"$3,204",12,"$1,473.36",$122.78,$267,$0,0


In [38]:
for fichier in fichiers_csv:
    df = pd.read_csv(fichier)

    print("\n", fichier.name)
    print(df.duplicated().sum())


 Sales 2020 T1.csv
0

 Sales 2020 T2.csv
0

 Sales 2020 T3.csv
0

 Sales 2021 T1.csv
0

 Sales 2021 T2.csv
0

 Sales 2021 T3.csv
0


## Activité 3.3 : valeurs manquantes

In [11]:
# compter les valeurs manquantes

for fichier in fichiers_csv:
    df = pd.read_csv(fichier)

    print("\n", fichier.name)
    print(df.isna().sum())


 Sales 2020 T1.csv
IdDate                  0
IdVente                 0
IdBoutique              0
IdProduit               0
IdPromotion             0
Montant du retour       0
Quantité de retour      0
Montant des ventes      0
Quantité de vente       0
Coût total              0
Coût unitaire           0
Prix unitaire           0
Montant de la remise    0
Quantité de remise      0
dtype: int64

 Sales 2020 T2.csv
IdDate                  0
IdVente                 0
IdBoutique              0
IdProduit               0
IdPromotion             0
Montant du retour       0
Quantité de retour      0
Montant des ventes      0
Quantité de vente       0
Coût total              0
Coût unitaire           0
Prix unitaire           0
Montant de la remise    0
Quantité de remise      0
dtype: int64

 Sales 2020 T3.csv
IdDate                  0
IdVente                 0
IdBoutique              0
IdProduit               0
IdPromotion             0
Montant du retour       0
Quantité de retour      0
Mont

# Prétraitement

# Normalisation des noms

In [19]:
import unicodedata
import re

def nettoyer_colonne(nom):
    nom = nom.strip().lower()
    nom = unicodedata.normalize("NFKD", nom)
    nom = "".join(c for c in nom if not unicodedata.combining(c))
    nom = re.sub(r"[^a-z0-9]+", "_", nom)
    return nom.strip("_")

In [20]:
# Renommer les identifiants
nouveaux_noms = {
    "iddate": "id_date",
    "idvente": "id_vente",
    "idboutique": "id_boutique",
    "idproduit": "id_produit",
    "idpromotion": "id_promotion"
}

df_exploration.columns = [
    nouveaux_noms.get(nettoyer_colonne(col), nettoyer_colonne(col))
    for col in df_exploration.columns
]

print(df_exploration.columns.tolist())

['id_date', 'id_vente', 'id_boutique', 'id_produit', 'id_promotion', 'montant_du_retour', 'quantite_de_retour', 'montant_des_ventes', 'quantite_de_vente', 'cout_total', 'cout_unitaire', 'prix_unitaire', 'montant_de_la_remise', 'quantite_de_remise']


In [21]:
df_exploration = df_exploration.rename(columns={
    "montant_du_retour": "montant_retour",
    "quantite_de_retour": "quantite_retour",
    "montant_des_ventes": "montant_ventes",
    "quantite_de_vente": "quantite_vente",
    "montant_de_la_remise": "montant_remise",
    "quantite_de_remise": "quantite_remise"
})

print(df_exploration.columns.tolist())

['id_date', 'id_vente', 'id_boutique', 'id_produit', 'id_promotion', 'montant_retour', 'quantite_retour', 'montant_ventes', 'quantite_vente', 'cout_total', 'cout_unitaire', 'prix_unitaire', 'montant_remise', 'quantite_remise']


In [24]:
# Vérification de l'unicité des noms

print("Noms uniques :", df_exploration.columns.is_unique)

Noms uniques : True


In [25]:
# Création de la fonction de renommage

def renommer_colonnes(df):
    df = df.copy()

    df.columns = [
        nouveaux_noms.get(nettoyer_colonne(col), nettoyer_colonne(col))
        for col in df.columns
    ]

    df = df.rename(columns={
        "montant_du_retour": "montant_retour",
        "quantite_de_retour": "quantite_retour",
        "montant_des_ventes": "montant_ventes",
        "quantite_de_vente": "quantite_vente",
        "montant_de_la_remise": "montant_remise",
        "quantite_de_remise": "quantite_remise"
    })

    return df

In [27]:
for fichier in fichiers_csv:
    df = pd.read_csv(fichier)
    df = renommer_colonnes(df)

    print(fichier.name, ":", df.columns.is_unique)

Sales 2020 T1.csv : True
Sales 2020 T2.csv : True
Sales 2020 T3.csv : True
Sales 2021 T1.csv : True
Sales 2021 T2.csv : True
Sales 2021 T3.csv : True


In [28]:
display(df_exploration.head())

,id_date,id_vente,id_boutique,id_produit,id_promotion,montant_retour,quantite_retour,montant_ventes,quantite_vente,cout_total,cout_unitaire,prix_unitaire,montant_remise,quantite_remise
0,1/1/2020 0:00,32306,248,910,10,$0,0,$754.80,8,$416,$52,$102,$61.20,3
1,1/1/2020 0:00,39789,212,514,10,$0,0,"$1,062.60",16,$365.76,$22.86,$69,$41.40,3
2,1/1/2020 0:00,89504,206,266,10,$0,0,"$2,392",8,"$1,219.52",$152.44,$299,$0,0
3,1/1/2020 0:00,123539,219,266,10,$0,0,"$2,212.60",8,"$1,219.52",$152.44,$299,$179.40,3
4,1/1/2020 0:00,134095,202,768,10,$0,0,$128.88,8,$73.04,$9.13,$17.90,$14.32,4


In [33]:
# Vérification des cohérences des colonnes

reference = renommer_colonnes(
    pd.read_csv(fichiers_csv[0])
).columns.tolist()

# Comparaison des six fichiers
for fichier in fichiers_csv:
    df = pd.read_csv(fichier)
    df = renommer_colonnes(df)

    print(fichier.name, df.columns.tolist() == reference)
                             

Sales 2020 T1.csv True
Sales 2020 T2.csv True
Sales 2020 T3.csv True
Sales 2021 T1.csv True
Sales 2021 T2.csv True
Sales 2021 T3.csv True


# Vérification des types

In [35]:
df = renommer_colonnes(pd.read_csv(fichiers_csv[0]))

df.dtypes

id_date              str
id_vente           int64
id_boutique        int64
id_produit         int64
id_promotion       int64
montant_retour       str
quantite_retour    int64
montant_ventes       str
quantite_vente     int64
cout_total           str
cout_unitaire        str
prix_unitaire        str
montant_remise       str
quantite_remise    int64
dtype: object